# Block-to-PUMA / POWPUMA Crosswalk Builder

Produces six output crosswalk CSVs for the two PUMA vintages (2019 = 2010-based PUMAs used for ACS 2012–2021; 2024 = 2020-based PUMAs used for ACS 2022+):

| Output file | Description |
|---|---|
| `stpuma_to_county_2019.csv` | PUMA → county name + FIPS (2019 vintage) |
| `stpuma_to_county_2024.csv` | PUMA → county name + FIPS (2024 vintage) |
| `powstpuma_to_county_2019.csv` | POWPUMA → county name(s) + FIPS (2019 vintage) |
| `powstpuma_to_county_2024.csv` | POWPUMA → county name(s) + FIPS (2024 vintage) |
| `blocks20{10,20}_x_puma{2019,2024}_xw.csv` | Block → PUMA (spatial join) |
| `blocks20{10,20}_x_powpuma{2019,2024}_xw.csv` | Block → POWPUMA (spatial join, requires POWPUMA shapefiles built in §3) |

**Workflow:**
1. Setup — imports and paths
2. ACS native crosswalks — parse official Excel code lists / correspondences for each vintage
3. Block → PUMA — spatial join of census blocks to PUMA polygons
4. POWPUMA shapefiles + Block → POWPUMA — dissolve PUMAs to POWPUMA polygons, then spatial join



## 0. Setup

In [113]:
import pathlib
import pandas as pd
import os
import geopandas as gpd

In [83]:
M_DRIVE = pathlib.Path('/Volumes/Data/Models') if os.name != 'nt' else pathlib.Path('M:/')
target_path = M_DRIVE / 'Crosswalks/geo/block-to-puma'
shapefile_path = M_DRIVE / 'Data/GIS/PUMA_Shapefiles'

data_base_path = M_DRIVE / 'Data/Census/PUMS'
ca_fips_path = M_DRIVE / 'Crosswalks/geo/st06_ca_cou2020.txt'

acs_code_list_2024 = M_DRIVE / 'Crosswalks/Census/PUMS/POWPUMA/ACSPUMS2024CodeLists.xls'
acs_code_list_2019 = M_DRIVE / 'Crosswalks/Census/PUMS/POWPUMA/ACSPUMS2019CodeLists.xls'
acs_code_list_2024


WindowsPath('M:/Crosswalks/Census/PUMS/POWPUMA/ACSPUMS2024CodeLists.xls')

In [84]:
ca_county_fips = pd.read_csv(ca_fips_path,sep='|',dtype=str)
ca_county_fips['STCOUNTY'] = ca_county_fips.STATEFP + ca_county_fips.COUNTYFP
ca_county_fips['county_name'] = ca_county_fips.COUNTYNAME#.str.replace(' County','')
ca_county_fips_map= ca_county_fips.set_index('STCOUNTY').county_name


### Shapefile paths

In [85]:
# Census block paths for the bay area only
blocks_2010_path = M_DRIVE / 'Data/GIS layers/Census/blocks/2010/blocks_bayarea_2010_geoid.shp'
blocks_2020_path = M_DRIVE / 'Data/GIS layers/Census/blocks/2020/blocks_bayarea_2020_geoid.shp'

# These are for CA as a whole
blocks_2010_path = M_DRIVE / 'Data/GIS layers/Census/blocks/tl_2010_06_tabblock10/tl_2010_06_tabblock10.shp'
blocks_2020_path = M_DRIVE / 'Data/GIS layers/Census/blocks/tl_2020_06_tabblock20/tl_2020_06_tabblock20.shp'

In [86]:
# PUMA paths

puma_2024_path = M_DRIVE /  'Data/GIS/PUMA_Shapefiles/puma_2024/tl_2024_06_puma20.shp'
#puma_2019_path = M_DRIVE / 'Data/GIS/PUMA_Shapefiles/puma_2019/tl_2012_06_puma10.shp'
puma_2019_path = M_DRIVE / 'Data/GIS/PUMA_Shapefiles/PUMA_2019/tl_2019_06_puma10.shp'
puma_2019_path


WindowsPath('M:/Data/GIS/PUMA_Shapefiles/PUMA_2019/tl_2019_06_puma10.shp')

## 2. Block → PUMA Spatial Join

Load the CA census block shapefiles (2010 and 2020 vintages) and PUMA shapefiles (2019 and 2024 vintages), then assign each block to its containing PUMA via point-in-polygon join. Used as the base crosswalk for LODES block geocodes. We note that in many cases we could do a tabular join based on PUMA names and typical fips string embeddings. But given the variability on PUMA name construction, and complex relation to counties, we use the spatial join approach instead.

### Census Blocks

In [87]:
blocks_2010 = gpd.read_file(blocks_2010_path)
blocks_2020 = gpd.read_file(blocks_2020_path)


In [88]:
blocks_2010['geom_pt'] = blocks_2010.representative_point()
blocks_2020['geom_pt'] = blocks_2020.representative_point()

blocks_2010.set_geometry('geom_pt', inplace=True)
blocks_2020.set_geometry('geom_pt', inplace=True)


## PUMAs

In [89]:
puma_2019 = gpd.read_file(puma_2019_path)
puma_2024 = gpd.read_file(puma_2024_path)

puma_2019 = puma_2019.to_crs(blocks_2010.crs)
puma_2024 = puma_2024.to_crs(blocks_2020.crs)

In [90]:
# this one was split in the 2020 vintage to 0611302, 0611301
puma_2019[puma_2019.GEOID10=='0611300']

,STATEFP10,PUMACE10,GEOID10,NAMELSAD10,MTFCC10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,geometry
85,06,11300,0611300,"Yolo County--Davis, Woodland & West Sacramento...",G6120,S,2628144759,22878428,+38.6795955,-121.9024413,"POLYGON ((-121.75694 38.52957, -121.75707 38.5..."


In [91]:
%%time 
blocks_2010_x_puma_2019 = gpd.sjoin(blocks_2010, puma_2019, how='left', predicate='within')
blocks_2020_x_puma_2024 = gpd.sjoin(blocks_2020, puma_2024, how='left', predicate='within')
blocks_2020_x_puma_2019 = gpd.sjoin(blocks_2020, puma_2019, how='left', predicate='within')


CPU times: total: 6 s
Wall time: 6.08 s


In [92]:
# blocks 2010 to puma 2019 (which is pumas from about 2012-2021)

blocks2010_x_puma2019_xw = blocks_2010_x_puma_2019[['GEOID10_left', 'GEOID10_right']].rename(columns={'GEOID10_left': 'block10_id', 'GEOID10_right': 'puma19_id','counties':'res_counties'})

# blocks 2020 to puma 2024
blocks2020_x_puma2024_xw = blocks_2020_x_puma_2024[['GEOID20_left', 'GEOID20_right']].rename(columns={'GEOID20_left': 'block20_id', 'GEOID20_right': 'puma24_id','counties':'res_counties'})

# blocks 2020 to puma 2012 / 2010
blocks2020_x_puma2019_xw = blocks_2020_x_puma_2019[['GEOID20', 'GEOID10']].rename(columns={'GEOID20': 'block20_id', 'GEOID10': 'puma19_id','counties':'res_counties'})
blocks_2010_x_puma_2019

,STATEFP10_left,COUNTYFP10,TRACTCE10,BLOCKCE10,GEOID10_left,NAME10,MTFCC10_left,UR10,UACE10,UATYP10,...,STATEFP10_right,PUMACE10,GEOID10_right,NAMELSAD10,MTFCC10_right,FUNCSTAT10_right,ALAND10_right,AWATER10_right,INTPTLAT10_right,INTPTLON10_right
0,06,001,410000,2024,060014100002024,Block 2024,G5040,U,78904,U,...,06,00103,0600103,Alameda County (Northeast)--Oakland (East) & P...,G6120,S,7.147193e+07,169613.0,+37.8041308,-122.1916059
1,06,001,407300,2017,060014073002017,Block 2017,G5040,U,78904,U,...,06,00104,0600104,Alameda County (North Central)--Oakland City (...,G6120,S,2.400298e+07,239048.0,+37.7620457,-122.1903690
2,06,001,408100,2011,060014081002011,Block 2011,G5040,U,78904,U,...,06,00103,0600103,Alameda County (Northeast)--Oakland (East) & P...,G6120,S,7.147193e+07,169613.0,+37.8041308,-122.1916059
3,06,001,401000,2002,060014010002002,Block 2002,G5040,U,78904,U,...,06,00102,0600102,Alameda County (Northwest)--Oakland (Northwest...,G6120,S,4.031090e+07,11207033.0,+37.8128270,-122.2820917
4,06,001,406100,1005,060014061001005,Block 1005,G5040,U,78904,U,...,06,00102,0600102,Alameda County (Northwest)--Oakland (Northwest...,G6120,S,4.031090e+07,11207033.0,+37.8128270,-122.2820917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710140,06,115,041100,4004,061150411004004,Block 4004,G5040,R,None,None,...,06,10100,0610100,Sutter & Yuba Counties--Yuba City PUMA,G6120,S,3.197003e+09,46298586.0,+39.1652663,-121.5020331
710141,06,115,041100,3015,061150411003015,Block 3015,G5040,R,None,None,...,06,10100,0610100,Sutter & Yuba Counties--Yuba City PUMA,G6120,S,3.197003e+09,46298586.0,+39.1652663,-121.5020331
710142,06,115,041100,2032,061150411002032,Block 2032,G5040,R,None,None,...,06,10100,0610100,Sutter & Yuba Counties--Yuba City PUMA,G6120,S,3.197003e+09,46298586.0,+39.1652663,-121.5020331
710143,06,115,041100,2063,061150411002063,Block 2063,G5040,R,None,None,...,06,10100,0610100,Sutter & Yuba Counties--Yuba City PUMA,G6120,S,3.197003e+09,46298586.0,+39.1652663,-121.5020331


In [93]:
# Add block's own county from the block FIPS prefix (doesn't require ACS lookups)
for df in [blocks2010_x_puma2019_xw, blocks2020_x_puma2024_xw, blocks2020_x_puma2019_xw]:
    block_col = df.filter(regex='block').columns[0]
    df['blk_county_name'] = df[block_col].str.slice(0, 5).map(ca_county_fips_map)

# NOTE: PUMS specific ACS-based 'counties' column (PUMA county grouping) is added and exported
# in the cell at the end of §1, after the PUMA→county lookup series are defined.


## 1. ACS Native Crosswalks (PUMA ↔ County, POWPUMA ↔ County)

Parse the ACS Excel code list workbooks to extract the official PUMA→POWPUMA→County relationships for each vintage. These are tabular lookups derived directly from Census documentation — no spatial processing needed.

ACS releases the crosswalks in two formats depending on vintage:
- **2019 (2010-based PUMAs, ACS 2012–2021):** two separate sheets — one for PUMA↔POWPUMA, one for County↔POWPUMA
- **2024 (2020-based PUMAs, ACS 2022+):** a single "Geography Lookup List" sheet with all relationships

### 1a. 2019 vintage (ACS 2012–2021)

(Note: PUMAs are redrawn after each decennial census. The 2012 vintage reflects 2010 counts and is identical across all ACS releases through 2021.)

The ACS 2019 code list provides two separate POWPUMA-related crosswalks:
- A sheet relating **PUMAs → POWPUMAs** (unit: PUMAs)
- A sheet relating **Counties → POWPUMAs** (unit: many-to-many)

For larger counties the relation is 1:1; smaller counties are grouped into a single larger POWPUMA.

In [94]:
# connect to the workbook
wb2019 = pd.ExcelFile(acs_code_list_2019)


# Load PUMA-to-POWPUMA mapping from ACS 2019 code list
puma_to_powpuma_cols_2019 = {
    'State of Residence (ST)': 'STATE', 
    'PUMA': 'PUMA5',
    'Place of Work State (POWSP) or Migration State (MIGSP)': 'POWSP3',
    'POWPUMA or MIGPUMA*': 'POWPUMA5'
}

# get the PUMA-to-POWPUMA mapping sheet, skipping header and footer rows, and renaming columns
puma_to_powpuma_2019 = wb2019.parse('PUMA_POWPUMA_MIGPUMA', skiprows=3, skipfooter=39, dtype=str)
puma_to_powpuma_2019.rename(columns=puma_to_powpuma_cols_2019, inplace=True)
puma_to_powpuma_2019['STPUMA'] = puma_to_powpuma_2019['STATE'] + puma_to_powpuma_2019['PUMA5']
puma_to_powpuma_2019['POWSTPUMA'] = puma_to_powpuma_2019['POWSP3'] + puma_to_powpuma_2019['POWPUMA5']

print(f"Loaded {len(puma_to_powpuma_2019)} PUMA-POWPUMA relationships")
puma_to_powpuma_2019.head()

Loaded 2378 PUMA-POWPUMA relationships


,STATE,PUMA5,POWSP3,POWPUMA5,STPUMA,POWSTPUMA
0,01,00100,001,00190,0100100,00100190
1,01,00200,001,00290,0100200,00100290
2,01,00301,001,00290,0100301,00100290
3,01,00302,001,00290,0100302,00100290
4,01,00400,001,00400,0100400,00100400


In [95]:
# Load county-to-POWPUMA mapping for geographic labels

powpuma_to_county_2019_cols = {
    'FIPS State Code': 'STATE_CODE',
    'FIPS County Code': 'COUNTY_CODE', 
    'Area Name': 'COUNTY_NAME',
    'MIGSP Migration (3-digit) and POWSP (3-digit) State Code': 'POWSP3',
    'MIGPUMA\nPOWPUMA': 'POWPUMA5'
}


# get the county-to-POWPUMA mapping sheet, skipping header and footer rows, and renaming columns
powpuma_to_county_2019 = wb2019.parse('MIGPUMA-POWPUMA 2019 Lookup', skiprows=3, skipfooter=20, dtype=str)
powpuma_to_county_2019.rename(columns=powpuma_to_county_2019_cols, inplace=True)
powpuma_to_county_2019['STCOUNTY'] = powpuma_to_county_2019['STATE_CODE'] + powpuma_to_county_2019['COUNTY_CODE']
powpuma_to_county_2019['POWSTPUMA'] = powpuma_to_county_2019['POWSP3'] + powpuma_to_county_2019['POWPUMA5']

print(f"Loaded {len(powpuma_to_county_2019)} county-POWPUMA relationships")
powpuma_to_county_2019.head()

Loaded 3220 county-POWPUMA relationships


,STATE_CODE,COUNTY_CODE,COUNTY_NAME,POWSP3,POWPUMA5,STCOUNTY,POWSTPUMA
0,01,001,Autauga County,001,02090,01001,00102090
1,01,003,Baldwin County,001,02600,01003,00102600
2,01,005,Barbour County,001,02400,01005,00102400
3,01,007,Bibb County,001,01700,01007,00101700
4,01,009,Blount County,001,00800,01009,00100800


In [96]:
# Create POWSTPUMA-to-county name lookup for geographic labeling of POWSTPUMAs. Note that some POWSTPUMAs span multiple counties, so we will concatenate county names where needed.

puma_to_powpuma_county_2019 = pd.merge(
    puma_to_powpuma_2019, 
    powpuma_to_county_2019, 
    on=['POWPUMA5', 'POWSP3', 'POWSTPUMA'], 
    how='inner'
)

## Powstpuma to county mappings
# First mapping POWSTPUMA to county names, concatenating where needed, and removing ' County' suffix for cleaner labels. 
powpuma_to_county_2019_xw1 = puma_to_powpuma_county_2019.groupby('POWSTPUMA').COUNTY_NAME.apply(
    lambda x: f"{', '.join(sorted(set(x.str.replace(' County', ''))))} County"
)

# Second mapping POWSTPUMA to concatenated FIPS codes for reference
powpuma_to_county_2019_xw2 = puma_to_powpuma_county_2019.groupby('POWSTPUMA').STCOUNTY.apply(
    lambda x: f"{', '.join(sorted(set(x)))}"
)

# combine the two mappings into a single DataFrame and write to CSV for use in geographic labeling of POWSTPUMAs
powpuma_to_county_2019_xw = pd.concat([powpuma_to_county_2019_xw1,powpuma_to_county_2019_xw2],axis=1,keys=['counties','fips'])

# write to csv
powpuma_to_county_2019_xw.reset_index().to_csv(target_path / 'powstpuma_to_county_2019.csv')

In [97]:
puma_to_powpuma_county_2019.head()

,STATE,PUMA5,POWSP3,POWPUMA5,STPUMA,POWSTPUMA,STATE_CODE,COUNTY_CODE,COUNTY_NAME,STCOUNTY
0,01,00100,001,00190,0100100,00100190,01,033,Colbert County,01033
1,01,00100,001,00190,0100100,00100190,01,057,Fayette County,01057
2,01,00100,001,00190,0100100,00100190,01,059,Franklin County,01059
3,01,00100,001,00190,0100100,00100190,01,075,Lamar County,01075
4,01,00100,001,00190,0100100,00100190,01,077,Lauderdale County,01077


In [98]:
## stpuma to county mappings

# First mapping POWSTPUMA to county names, sorting, and concatenating
puma_to_county_2019_xw1 = puma_to_powpuma_county_2019.groupby('STPUMA').COUNTY_NAME.apply(
    lambda x: f"{', '.join(sorted(set(x.str.replace(' County', ''))))} County"
)
# Second mapping POWSTPUMA to concatenated FIPS codes for reference
puma_to_county_2019_xw2 = puma_to_powpuma_county_2019.groupby('STPUMA').STCOUNTY.apply(
    lambda x: f"{', '.join(sorted(set(x)))}"
)

# combine the two mappings into a single DataFrame and write to CSV for use in geographic labeling of POWSTPUMAs
puma_to_county_2019_xw = pd.concat([puma_to_county_2019_xw1,puma_to_county_2019_xw2],axis=1,keys=['counties','fips'])

# write to csv
puma_to_county_2019_xw.reset_index().to_csv(target_path / 'stpuma_to_county_2019.csv')

### 1b. 2024 vintage (ACS 2022+)

Unlike the 2019 code list, the 2024 edition consolidates everything into a single "Geography Lookup List" sheet: County → PUMA → POWPUMA in one many-to-many table.

In [99]:
# Load PUMA-to-POWPUMA mapping from ACS 2024 code list

# columns are a bit unruly in the original so we simplify and rename for easier use
pums2024_cols = {'State FIPS Code \n(2-digit)':'STATE', 'County FIPS Code\n(3-digit)': 'COUNTY',
       'County or County Equivalent Name':'COUNTY_NAME', 'State Code\n(2-digit)': 'STATE_CODE',
       'Public Use Microdata Area Code\n(PUMA 5-digit)':'PUMA5',
       'Migration State Code\n(MIGSP 3-digit) and\nPlace of Work State Code\n(POWSP 3-digit)':'POWSP3',
       'Migration Public Use Microdata Area Code\n(MIGPUMA 5-digit) and\nPlace of Work Public Use Microdata Area Code\n(POWPUMA 5-digit)': 'POWPUMA5'}


# load the sheet
puma_to_powpuma_2024 = pd.read_excel(acs_code_list_2024, sheet_name='Geography Lookup List', skiprows=3,dtype=str)

puma_to_powpuma_2024.rename(columns=pums2024_cols, inplace=True)
puma_to_powpuma_2024['STPUMA'] = puma_to_powpuma_2024['STATE'] + puma_to_powpuma_2024['PUMA5']
puma_to_powpuma_2024['STCOUNTY'] = puma_to_powpuma_2024['STATE'] + puma_to_powpuma_2024['COUNTY']
puma_to_powpuma_2024['POWSTPUMA'] = puma_to_powpuma_2024['POWSP3'] + puma_to_powpuma_2024['POWPUMA5']
puma_to_powpuma_xw_2024 = puma_to_powpuma_2024.groupby('STPUMA').POWSTPUMA.first()

print(f"Loaded {len(puma_to_powpuma_2024)} PUMA-POWPUMA relationships (2024 vintage)")
puma_to_powpuma_2024.head()


Loaded 4553 PUMA-POWPUMA relationships (2024 vintage)


,STATE,COUNTY,COUNTY_NAME,STATE_CODE,PUMA5,POWSP3,POWPUMA5,STPUMA,STCOUNTY,POWSTPUMA
0,01,001,Autauga County,01,01700,001,01700,0101700,01001,00101700
1,01,003,Baldwin County,01,02701,001,02700,0102701,01003,00102700
2,01,003,Baldwin County,01,02702,001,02700,0102702,01003,00102700
3,01,005,Barbour County,01,02500,001,02500,0102500,01005,00102500
4,01,007,Bibb County,01,01100,001,01100,0101100,01007,00101100


In [100]:

# Create POWSTPUMA shapefile for 2022 vintage (PUMS 2022+)
puma_2024_col_keep = ['GEOID20', 'STATEFP20', 'ALAND20', 'AWATER20', 'geometry', 'POWSTPUMA']

# First mapping POWSTPUMA to county names, concatenating where needed, and removing ' County' suffix for cleaner labels. 
powpuma_to_county_2024_xw1 = puma_to_powpuma_2024.groupby('POWSTPUMA').apply(
    lambda x: f"{', '.join(sorted(set(x.COUNTY_NAME.str.replace(' County', ''))))} County"
)

# Second mapping POWSTPUMA to concatenated FIPS codes for reference
powpuma_to_county_2024_xw2 = puma_to_powpuma_2024.groupby('POWSTPUMA').apply(
    lambda x: f"{', '.join(sorted(set(x.STCOUNTY)))}"
)

# combine the two mappings into a single DataFrame and write to CSV for use in geographic labeling of POWSTPUMAs
powpuma_to_county_2024_xw = pd.concat([powpuma_to_county_2024_xw1,powpuma_to_county_2024_xw2],keys=['counties','fips'],axis=1)
print(f"Created county labels for {len(powpuma_to_county_2024_xw1)} POWSTPUMAs")

# write to csv
powpuma_to_county_2024_xw.reset_index().to_csv(target_path / 'powstpuma_to_county_2024.csv')

Created county labels for 1027 POWSTPUMAs


C:\Users\aolsen\AppData\Local\Temp\ipykernel_10332\711681669.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  powpuma_to_county_2024_xw1 = puma_to_powpuma_2024.groupby('POWSTPUMA').apply(
C:\Users\aolsen\AppData\Local\Temp\ipykernel_10332\711681669.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  powpuma_to_county_2024_xw2 = puma_to_powpuma_2024.groupby('POWSTPUMA').apply(


In [101]:
# First mapping STPUMA to county names, concatenating where needed, and removing ' County' suffix for cleaner labels. 
stpuma_to_county_2024_xw1 = puma_to_powpuma_2024.groupby('STPUMA').apply(
    lambda x: f"{', '.join(sorted(set(x.COUNTY_NAME.str.replace(' County', ''))))} County"
)

# Second mapping STPUMA to concatenated FIPS codes for reference
stpuma_to_county_2024_xw2 = puma_to_powpuma_2024.groupby('STPUMA').apply(
    lambda x: f"{', '.join(sorted(set(x.STCOUNTY)))}"
)

# combine the two mappings into a single DataFrame and write to CSV for use in geographic labeling of STPUMAs
stpuma_to_county_2024_xw = pd.concat([stpuma_to_county_2024_xw1,stpuma_to_county_2024_xw2],keys=['counties','fips'],axis=1)
print(f"Created county labels for {len(stpuma_to_county_2024_xw1)} STPUMAs")

# write to csv
stpuma_to_county_2024_xw.reset_index().to_csv(target_path / 'stpuma_to_county_2024.csv')

Created county labels for 2486 STPUMAs


C:\Users\aolsen\AppData\Local\Temp\ipykernel_10332\3040605469.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stpuma_to_county_2024_xw1 = puma_to_powpuma_2024.groupby('STPUMA').apply(
C:\Users\aolsen\AppData\Local\Temp\ipykernel_10332\3040605469.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stpuma_to_county_2024_xw2 = puma_to_powpuma_2024.groupby('STPUMA').apply(


In [102]:
# check sf = the STPUMAs should start at 067507
stpuma_to_county_2024_xw[stpuma_to_county_2024_xw.counties=="San Francisco County"]

,counties,fips
STPUMA,,
0607507,San Francisco County,06075
0607508,San Francisco County,06075
0607509,San Francisco County,06075
0607510,San Francisco County,06075
0607511,San Francisco County,06075
0607512,San Francisco County,06075
0607513,San Francisco County,06075
0607514,San Francisco County,06075


In [103]:
# check sf = the STPUMAs should start at 067507
powpuma_to_county_2024_xw[powpuma_to_county_2024_xw.counties=="San Francisco County"]

,counties,fips
POWSTPUMA,,
00607500,San Francisco County,06075


In [104]:
# Add ACS-based county grouping to block→PUMA crosswalks and export.
# Runs here (end of §1) so that the PUMA→county lookup series are defined.

blocks2010_x_puma2019_xw['res_counties'] = blocks2010_x_puma2019_xw['puma19_id'].map(puma_to_county_2019_xw1)
blocks2020_x_puma2019_xw['res_counties'] = blocks2020_x_puma2019_xw['puma19_id'].map(puma_to_county_2019_xw1)
blocks2020_x_puma2024_xw['res_counties'] = blocks2020_x_puma2024_xw['puma24_id'].map(stpuma_to_county_2024_xw1)

blocks2010_x_puma2019_xw.to_csv(target_path / 'blocks2010_x_puma2019_xw.csv', index=False)
blocks2020_x_puma2024_xw.to_csv(target_path / 'blocks2020_x_puma2024_xw.csv', index=False)
blocks2020_x_puma2019_xw.to_csv(target_path / 'blocks2020_x_puma2019_xw.csv', index=False)
print("Exported block→PUMA crosswalks")


Exported block→PUMA crosswalks


In [105]:
puma_to_county_2019_xw1[puma_to_county_2019_xw1.str.contains('San Ben')]

STPUMA
0605301    Monterey, San Benito County
0605302    Monterey, San Benito County
0605303    Monterey, San Benito County
Name: COUNTY_NAME, dtype: object

## 3. POWPUMA Shapefiles + Block → POWPUMA Spatial Join

POWPUMAs don't ship as shapefiles — they need to be constructed by dissolving PUMA polygons on the POWSTPUMA code derived from the ACS crosswalk (§1 above). Once the POWPUMA polygons exist, the block → POWPUMA spatial join follows the same pattern as §2.

### 3a. Build POWPUMA polygons (dissolve PUMAs)

In [106]:
# Create POWSTPUMA shapefile for 2012 vintage (PUMS 2012-2021)
puma_2019_col_keep = ['GEOID10', 'STATEFP10', 'ALAND10', 'AWATER10', 'geometry', 'POWSTPUMA']

puma_2019['POWSTPUMA'] = puma_2019.GEOID10.map(puma_to_powpuma_2019.set_index('STPUMA')['POWSTPUMA'])

gdf_powstpuma_2019 = puma_2019[puma_2019_col_keep].dissolve('POWSTPUMA', as_index=False)
gdf_powstpuma_2019.head()

gdf_powstpuma_2019['counties'] = gdf_powstpuma_2019.POWSTPUMA.map(powpuma_to_county_2019_xw.counties)

print(f"Created {len(gdf_powstpuma_2019)} POWSTPUMA polygons for 2012 vintage")
gdf_powstpuma_2019.head()

Created 41 POWSTPUMA polygons for 2012 vintage


,POWSTPUMA,geometry,GEOID10,STATEFP10,ALAND10,AWATER10,counties
0,00600100,"POLYGON ((-122.11747 37.50699, -122.11828 37.5...",0600108,06,176597287,12603090,Alameda County
1,00600300,"POLYGON ((-118.33758 36.6548, -118.33773 36.65...",0600300,06,49906032250,552479248,"Alpine, Amador, Calaveras, Inyo, Mariposa, Mon..."
2,00600700,"POLYGON ((-121.3546 39.84051, -121.35456 39.84...",0600701,06,1319401062,11274683,Butte County
3,00601100,"POLYGON ((-122.19917 40.37794, -122.19913 40.3...",0601100,06,22255986864,155477970,"Colusa, Glenn, Tehama, Trinity County"
4,00601300,"POLYGON ((-121.69733 37.78265, -121.69842 37.7...",0601301,06,52812877,19406513,Contra Costa County


In [107]:

# Map, dissolve, and add county attributes
puma_2024['POWSTPUMA'] = puma_2024.GEOID20.map(puma_to_powpuma_xw_2024)
gdf_powstpuma_2024 = puma_2024[puma_2024_col_keep].dissolve('POWSTPUMA', as_index=False)
gdf_powstpuma_2024['counties'] = gdf_powstpuma_2024.POWSTPUMA.map(powpuma_to_county_2024_xw1)

print(f"Created {len(gdf_powstpuma_2024)} POWSTPUMA polygons for 2022 vintage")
gdf_powstpuma_2024.head()

Created 41 POWSTPUMA polygons for 2022 vintage


,POWSTPUMA,geometry,GEOID20,STATEFP20,ALAND20,AWATER20,counties
0,00600100,"POLYGON ((-122.11747 37.50699, -122.11828 37.5...",0600101,06,31654342,8608712,Alameda County
1,00600300,"POLYGON ((-120.10345 37.22442, -120.1056 37.22...",0600300,06,49906109755,552655522,"Alpine, Amador, Calaveras, Inyo, Mariposa, Mon..."
2,00600700,"POLYGON ((-121.30711 39.4978, -121.3102 39.497...",0600700,06,4238545149,105204062,Butte County
3,00601100,"POLYGON ((-122.00005 38.92554, -122.00041 38.9...",0601100,06,22255922107,155542780,"Colusa, Glenn, Tehama, Trinity County"
4,00601300,"POLYGON ((-121.93281 37.7252, -121.93338 37.72...",0601301,06,52895802,19395052,Contra Costa County


In [108]:
# Export shapefiles 
gdf_powstpuma_2019.to_file(shapefile_path / 'powstpuma_2019.shp')
gdf_powstpuma_2024.to_file(shapefile_path / 'powstpuma_2024.shp')

print(f"Exported POWSTPUMA shapefiles to: {shapefile_path}")
print(f"  - powstpuma_2019.shp ({len(gdf_powstpuma_2019)} features)")
print(f"  - powstpuma_2024.shp ({len(gdf_powstpuma_2024)} features)")

Exported POWSTPUMA shapefiles to: M:\Data\GIS\PUMA_Shapefiles
  - powstpuma_2019.shp (41 features)
  - powstpuma_2024.shp (41 features)


### 3b. Spatial join: blocks → POWPUMA

In [109]:
# Finally, with POWPUMA shapefiles we can relate powpumas to census blocks


blocks_2010_x_powpuma_2019 = gpd.sjoin(blocks_2010, gdf_powstpuma_2019, how='left', predicate='within')
blocks_2020_x_powpuma_2024 = gpd.sjoin(blocks_2020, gdf_powstpuma_2024, how='left', predicate='within')
blocks_2020_x_powpuma_2019 = gpd.sjoin(blocks_2020, gdf_powstpuma_2019, how='left', predicate='within')



In [110]:
# blocks 2010 to puma 2012

blocks2010_x_powpuma2019_xw = blocks_2010_x_powpuma_2019[['GEOID10_left', 'POWSTPUMA','counties']].rename(columns={'GEOID10_left': 'block10_id','POWSTPUMA':'POWSTPUMA_2019','counties':'pow_counties'})

# blocks 2020 to puma 2022
blocks2020_x_powpuma2024_xw = blocks_2020_x_powpuma_2024[['GEOID20_left', 'POWSTPUMA','counties']].rename(columns={'GEOID20_left': 'block20_id','POWSTPUMA':'POWSTPUMA_2024','counties':'pow_counties'})

# blocks 2020 to puma 2012 / 2010
blocks2020_x_powpuma2019_xw = blocks_2020_x_powpuma_2019[['GEOID20', 'POWSTPUMA','counties']].rename(columns={'GEOID20': 'block20_id','POWSTPUMA':'POWSTPUMA_2019','counties':'pow_counties'})
blocks2020_x_powpuma2019_xw


,block20_id,POWSTPUMA_2019,pow_counties
0,060650406112004,00606500,Riverside County
1,060650402014001,00606500,Riverside County
2,061130101032011,00611300,Yolo County
3,061130115001002,00611300,Yolo County
4,061130107041021,00611300,Yolo County
...,...,...,...
519718,060830029372018,00608300,Santa Barbara County
519719,060510001011103,00600300,"Alpine, Amador, Calaveras, Inyo, Mariposa, Mon..."
519720,060590630072015,00605900,Orange County
519721,060379012091349,00603700,Los Angeles County


In [111]:
# add block's own county (from block FIPS prefix)
for df in [blocks2010_x_powpuma2019_xw, blocks2020_x_powpuma2024_xw, blocks2020_x_powpuma2019_xw]:
    block_col = df.filter(regex='block').columns[0]
    df['blk_county_name'] = df[block_col].str.slice(0, 5).map(ca_county_fips_map)

# add ACS-based county grouping — the county label(s) that come with the POWPUMA,
# which may aggregate multiple small counties into one POWPUMA
blocks2010_x_powpuma2019_xw['counties'] = blocks2010_x_powpuma2019_xw['POWSTPUMA_2019'].map(powpuma_to_county_2019_xw1)
blocks2020_x_powpuma2019_xw['counties'] = blocks2020_x_powpuma2019_xw['POWSTPUMA_2019'].map(powpuma_to_county_2019_xw1)
blocks2020_x_powpuma2024_xw['counties'] = blocks2020_x_powpuma2024_xw['POWSTPUMA_2024'].map(powpuma_to_county_2024_xw1)
blocks2020_x_powpuma2019_xw


,block20_id,POWSTPUMA_2019,pow_counties,blk_county_name,counties
0,060650406112004,00606500,Riverside County,Riverside County,Riverside County
1,060650402014001,00606500,Riverside County,Riverside County,Riverside County
2,061130101032011,00611300,Yolo County,Yolo County,Yolo County
3,061130115001002,00611300,Yolo County,Yolo County,Yolo County
4,061130107041021,00611300,Yolo County,Yolo County,Yolo County
...,...,...,...,...,...
519718,060830029372018,00608300,Santa Barbara County,Santa Barbara County,Santa Barbara County
519719,060510001011103,00600300,"Alpine, Amador, Calaveras, Inyo, Mariposa, Mon...",Mono County,"Alpine, Amador, Calaveras, Inyo, Mariposa, Mon..."
519720,060590630072015,00605900,Orange County,Orange County,Orange County
519721,060379012091349,00603700,Los Angeles County,Los Angeles County,Los Angeles County


In [ ]:

# write to target path 
blocks2010_x_powpuma2019_xw.to_csv(target_path / 'blocks2010_x_powpuma2019_xw.csv', header=True,index=False)    
blocks2020_x_powpuma2024_xw.to_csv(target_path / 'blocks2020_x_powpuma2024_xw.csv', header=True,index=False)
blocks2020_x_powpuma2019_xw.to_csv(target_path / 'blocks2020_x_powpuma2019_xw.csv', header=True,index=False)

